## Upload TIFF Pixel Masks as Prelabels

This notebook uploads externally-generated segmentation masks into Centaur and sets them as
**prelabels** — the starting annotations labelers see and correct — on a `pixel_seg` task.

Use it when your masks are produced outside Centaur (for example by your own inference pipeline)
and you already have one binary TIFF per label class per image frame.

The flow is four API calls per batch of masks:

1. `POST /masks/public/v1/generate-presigned-urls` — ask for upload URLs
2. `PUT <presigned url>` — upload the raw TIFF bytes to S3
3. `POST /masks/public/v1/commit` — validate the masks and get a `mask_uuid` for each
4. `PUT /cases/prelabels/public/v1/set` — attach those masks to the case as prelabels

This example uses:

- `requests` for API calls
- `tifffile` and `numpy` for a local pre-flight check on each mask, so bad files are caught before
  upload rather than in the commit response

Reference documentation: https://docs.centaurlabs.com/


In [ ]:
# Import required libraries
import math
from typing import Any, Dict, List

import numpy as np
import requests
import tifffile

In [ ]:
# API URLs
base_url = "https://api.centaurlabs.com"

auth_url = f"{base_url}/auth/public/v1/login"
presign_url = f"{base_url}/masks/public/v1/generate-presigned-urls"
commit_url = f"{base_url}/masks/public/v1/commit"
prelabels_url = f"{base_url}/cases/prelabels/public/v1/set"

In [ ]:
# Parameters for headers
centaur_email = "your_portal_email"  # Replace with your Centaur Email
centaur_api_key = "your_api_key_here"  # Replace with your Centaur API key
centaur_api_password = "your_api_password_here"  # Replace with your Centaur API password (not your portal password)

# Task parameters
project_id = 123  # Replace with your project ID
task_id = 456  # Replace with your task ID — the task must be of type pixel_seg

# Label classes.
#
# label_class_id is a PER-TASK ordinal assigned in the order the classes were created on the task:
# the first class is 0, the second is 1, and so on. It is not a global ID, and no public endpoint
# returns it, so record the creation order when you set the task up (or ask your Centaur contact).
#
# The mask endpoints take the integer id; the prelabel endpoint takes the class name. Keeping both
# in one mapping avoids mixing them up.
label_classes = {
    "liver": 0,  # Replace with your class names and their ordinals
    "kidney": 1,
}

### Describing your masks

Centaur needs three identifiers to place a mask, and they must agree with each other:

| Identifier | What it is |
|---|---|
| `case_id` | The case (also called `problem_id`) the mask belongs to |
| `frame_id` | Which frame *within* that case. Single-frame cases are `0` |
| `content_id` | The **asset** backing that frame |

`frame_id` and `content_id` describe the same frame from two different angles: the upload steps
identify the frame by `frame_id`, and the prelabel step identifies it by `content_id`. Centaur
resolves `content_id` back to a frame internally, so if the two disagree your mask will upload
successfully and then not be found when you set the prelabel.

Both come from the same place — `GET /results/public/v2/list?task_id=...&verbose=true` returns each
case together with its `content_metadata`, which carries the `content_id` for each frame in frame
order. For a multi-slice volume, confirm the slice ordering with your Centaur contact before a large
run rather than assuming it; a mask set that is off by one slice is easy to produce and hard to spot.

Fill in one entry per frame below.

In [ ]:
# One entry per (case, frame). `masks` maps a label class name to the TIFF file for that class.
# Omit a class entirely if there is no finding for it on that frame.
frames_to_upload: List[Dict[str, Any]] = [
    {
        "case_id": 789,  # Replace with your case ID
        "frame_id": 0,  # Replace with the frame index within the case
        "content_id": 27914771,  # Replace with the asset ID for that frame
        "masks": {
            "liver": "masks/case789_frame0_liver.tiff",
            "kidney": "masks/case789_frame0_kidney.tiff",
        },
    },
]

In [ ]:
# Helper to report an error response. Not every failure returns JSON — a 500 comes back as plain
# text, and calling .json() on it raises and hides the real status.
def error_detail(response):
    try:
        return f"{response.status_code} {response.json()}"
    except ValueError:
        return f"{response.status_code} {response.text[:300]}"


# Function to get the authentication token
def get_auth_token(email, password, key):
    payload = {"username": email, "api_password": password}
    headers = {"X-API-KEY": key}
    response = requests.post(auth_url, headers=headers, json=payload)
    if response.status_code == 200:
        return response.json().get("token")
    else:
        raise Exception(f"Authentication failed: {error_detail(response)}")


# Every call below sends BOTH the API key and the bearer token. The key alone returns 401.
def auth_headers(token, key):
    return {"Authorization": f"Bearer {token}", "X-API-KEY": key}

### Mask file requirements

Each TIFF must be:

- **single channel** (grayscale, not RGB)
- **binary** — `0` and `255`, or `0` and `1`. Any non-zero value is treated as part of the mask
- **exactly the frame's pixel dimensions**

Commit downloads and checks every mask against the frame's real dimensions, so a mismatch comes back
as a per-mask failure rather than a whole-request error. The check below catches the same problems
locally first, which is much faster than a round trip when you are preparing a large batch.

In [ ]:
# Function to check a mask file before uploading it
def validate_mask_file(path, expected_shape=None):
    """Check one mask TIFF. Returns its (height, width). Raises if it would be rejected on commit."""
    array = tifffile.imread(path)

    if array.ndim == 3 and array.shape[2] == 1:
        array = array[:, :, 0]
    if array.ndim != 2:
        raise Exception(f"{path}: mask must be single channel, got shape {array.shape}")

    if expected_shape is not None and array.shape != expected_shape:
        raise Exception(
            f"{path}: expected {expected_shape[1]}x{expected_shape[0]}, got {array.shape[1]}x{array.shape[0]}"
        )

    values = set(np.unique(array).tolist())
    if not (values <= {0, 1} or values <= {0, 255}):
        print(
            f"  warning: {path} is not strictly binary (values {sorted(values)[:5]}); non-zero pixels will be used"
        )

    if values == {0}:
        print(
            f"  note: {path} is empty — it will commit as 'no finding' for that class"
        )

    return array.shape

In [ ]:
# Function to request presigned upload URLs for one frame's masks
def generate_presigned_urls(token, key, frame):
    class_ids = [label_classes[name] for name in frame["masks"]]
    payload = {
        "project_id": project_id,
        "task_id": task_id,
        "masks": [
            {
                "problem_id": frame["case_id"],
                "frame_id": frame["frame_id"],
                "label_class_ids": class_ids,
                "category": "pre_labels",
                "mask_type": "tiff",
            },
        ],
    }
    response = requests.post(
        presign_url, headers=auth_headers(token, key), json=payload
    )
    if response.status_code != 200:
        raise Exception(f"Failed to generate presigned URLs: {error_detail(response)}")

    # The `uuid` in the response only names the upload path. The mask_uuid you need for prelabels is
    # generated later, by commit, so there is nothing to keep from here.
    return response.json()["masks"][0]["presigned_urls"]

In [ ]:
# Function to upload one mask's bytes to its presigned URL
def upload_mask(file_url, path):
    with open(path, "rb") as mask_file:
        data = mask_file.read()

    # The Content-Type is part of what the presigned URL is signed for, so it has to be exactly this
    # value — S3 rejects anything else, including image/tiff, with 403 SignatureDoesNotMatch.
    # No API key or bearer token on this call: the URL itself is the credential.
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    response = requests.put(file_url, data=data, headers=headers)
    if response.status_code != 200:
        raise Exception(
            f"Failed to upload {path}: {response.status_code} {response.text[:300]}"
        )

In [ ]:
# Function to commit uploaded masks. The public endpoint accepts at most 20 masks per call.
COMMIT_BATCH_SIZE = 20


def commit_masks(token, key, mask_requests):
    """Commit masks in batches. Returns {(case_id, frame_id, label_class_id): mask_uuid}."""
    committed = {}

    for start in range(0, len(mask_requests), COMMIT_BATCH_SIZE):
        batch = mask_requests[start : start + COMMIT_BATCH_SIZE]
        payload = {"task_id": task_id, "masks": [m["request"] for m in batch]}
        response = requests.post(
            commit_url, headers=auth_headers(token, key), json=payload
        )

        # 200 = every mask committed, 207 = some failed. Both return per-mask results.
        if response.status_code not in (200, 207):
            raise Exception(f"Commit failed: {error_detail(response)}")

        for result in response.json()["masks"]:
            info = result["mask_request_info"]
            identity = (info["problem_id"], info["frame_id"], info["label_class_id"])

            if not result["success"]:
                print(f"  FAILED {identity}: {result.get('error_message')}")
                continue

            # One entry per committed frame. An all-zero mask commits with a null uuid, which means
            # "no finding" for that class — there is nothing to set as a prelabel.
            mask_uuid = next(
                (
                    f["mask_uuid"]
                    for f in (result.get("frame_data") or [])
                    if f["mask_uuid"]
                ),
                None,
            )
            if mask_uuid is None:
                print(f"  empty mask {identity}: nothing to set as a prelabel")
                continue

            committed[identity] = mask_uuid

    return committed

In [ ]:
# Function to set the committed masks as prelabels.
#
# One entry in `labels` describes one frame: a (case_id, content_id) pair plus every class on that
# frame. Entries that share a case_id are merged into a single prelabel for that case, so send all of
# a case's frames together rather than one call per frame.
def set_prelabels(token, key, frames, committed):
    labels = []

    for frame in frames:
        answers = []
        for class_name in frame["masks"]:
            identity = (frame["case_id"], frame["frame_id"], label_classes[class_name])
            if identity in committed:
                answers.append(
                    {"answer_class": class_name, "mask_uuid": committed[identity]}
                )

        if answers:
            labels.append(
                {
                    "case_id": frame["case_id"],
                    "content_id": frame["content_id"],
                    "answer": answers,
                }
            )

    if not labels:
        print("No committed masks to set as prelabels")
        return

    response = requests.put(
        prelabels_url,
        headers=auth_headers(token, key),
        params={"task_id": task_id},
        json={"labels": labels},
    )
    if response.status_code != 200:
        raise Exception(f"Failed to set prelabels: {error_detail(response)}")

    print(
        f"Prelabels set for {len({label['case_id'] for label in labels})} case(s): {response.json()}"
    )

In [ ]:
# Main script to run the process
try:
    # Step 0: Authenticate
    centaur_token = get_auth_token(centaur_email, centaur_api_password, centaur_api_key)
    print("Authentication successful")

    # Step 1 and 2: request an upload URL for every mask and upload it
    mask_requests = []

    for frame in frames_to_upload:
        print(f"Case {frame['case_id']} frame {frame['frame_id']}:")
        presigned_urls = generate_presigned_urls(centaur_token, centaur_api_key, frame)
        url_by_class_id = {u["label_class_id"]: u["file_url"] for u in presigned_urls}

        frame_shape = None
        for class_name, path in frame["masks"].items():
            # All masks on a frame must be the same size as each other and as the frame itself
            frame_shape = validate_mask_file(path, expected_shape=frame_shape)

            file_url = url_by_class_id[label_classes[class_name]]
            upload_mask(file_url, path)
            print(f"  uploaded {class_name} from {path}")

            mask_requests.append(
                {
                    "request": {
                        "presigned_url": file_url,
                        "problem_id": frame["case_id"],
                        "label_class_id": label_classes[class_name],
                        "category": "pre_labels",
                        "frame_id": frame["frame_id"],
                    },
                }
            )

    # Step 3: commit the uploads and collect a mask_uuid for each
    batches = math.ceil(len(mask_requests) / COMMIT_BATCH_SIZE)
    print(f"Committing {len(mask_requests)} mask(s) in {batches} batch(es)")
    committed = commit_masks(centaur_token, centaur_api_key, mask_requests)
    print(f"Committed {len(committed)} mask(s)")

    # Step 4: attach them to their cases as prelabels
    set_prelabels(centaur_token, centaur_api_key, frames_to_upload, committed)

except Exception as e:
    print("An error occurred:", str(e))

### Troubleshooting

| Symptom | Cause |
|---|---|
| `401` on every call | Only the API key was sent. Every call needs the bearer token from `/auth/public/v1/login` as well |
| `403 SignatureDoesNotMatch` on the upload | The `Content-Type` on the `PUT` is not exactly `application/x-www-form-urlencoded` |
| `403` / `401` on presign or set prelabels | The API key needs the permission to set answers on the task. Ask your Centaur contact to check it |
| `400` naming a field on presign or commit | Presign takes `label_class_ids` (plural); commit takes `label_class_id` (singular). The response body names the field it rejected |
| `Request has answer class ids not in task` | A `label_class_id` the task does not define. They are per-task ordinals starting at `0` |
| `Image size mismatch, expected 512x512, got (256, 256)` | The mask is not the frame's pixel dimensions |
| `Image is not grayscale` | The TIFF has more than one channel |
| `Pixel mask for answer class '<name>' was not found` | The `content_id` in step 4 points at a different frame than the `frame_id` the mask was committed under |
| A class silently has no prelabel | Its mask was all zeros, which commits as "no finding" for that class |

Prelabels appear on a `pixel_seg` task the next time the case is opened for labeling.
